### **Setup: Imports und Initialisierung**

In [ ]:
import sys
import os
import ast
import json
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

sys.path.append(os.path.abspath('..'))

from quinex_application.quinex_processing.mongos_store import MongoStore
from quinex_application.quinex_processing.quinex_client import QuinexClient
from quinex_application.quinex_processing.quinex_postprocessing import QuinexPostprocessing
from quinex_application.postprocessing.spatial_normalization import SpatialNormalization
from quinex_application.quinex_processing.openalex_client import OpenAlexClient
from quinex_application.visualisation.metadata_visualisation import VisualisationMetadata

In [ ]:
quinex = QuinexClient()
mongodb = MongoStore(quinex, database_name="quinex_db_query4", collection_name="papers_2010")
qpost = QuinexPostprocessing()
spatial = SpatialNormalization()
oalex = OpenAlexClient()
metadata = VisualisationMetadata("../data/quinex/df_quinex_metadata_new.parquet")

### **Load Analysis from MongoDB into Metadata DF and Data DF**

In [ ]:
analysis = []
for year in range(2025, 2026):
    #analysis.append(f"papers_{year}")
    analysis.append(f"papers_{year}_closed")
    #analysis.append(f"papers_{year}_abstract_fallback")

#analysis += ["papers_wos", "papers_wos_closed", "papers_wos_abstract_fallback"]

meta_df = pd.read_parquet("../data/paper/paper_dataset.parquet")
df_path = "../data/quinex/df_quinex.parquet"
df_metadata_path = "../data/quinex/df_quinex_metadata_new.parquet"

for ana in analysis:
    a = mongodb.get_collection(ana)
    
    # if "_abstract_fallback" not in ana:
    #     df_quinex_meta = qpost.create_metadata_df(a, meta_df)
    #     qpost.save_df(df_quinex_meta, df_metadata_path)

    df_quinex = qpost.create_information_df(a, meta_df)
    print(len(df_quinex[df_quinex["DOI"] == "10.1016/j.esr.2025.101720"]))
    qpost.save_df(df_quinex, df_path)

## **Information Dataframe**

### **Rerun Spatial Normalization**

In [ ]:
df_quinex = pd.read_parquet("../data/quinex/df_quinex.parquet")

In [ ]:
df = spatial.get_geo_information_from_spatial_scope(
    df_quinex, 
    nbr_runs_per_model=1, 
    models = [
        "7 - Qwen3-Coder-30B-A3B-Instruct - A code model from August 2025", 
        "0 - Ministral-3-14B-Instruct-2512 - The latest Ministral from Dec.2.2025",
        ]
    )

In [ ]:
df_path = "../data/quinex/df_quinex.parquet"
df["spatial_scope_refined"] = df["spatial_scope_refined"].apply(
    lambda x: json.dumps(x) if isinstance(x, dict) else None
).astype("string")

df.to_parquet(df_path, index=False)

### **Rerun Quantity Parsing**

In [ ]:
df_quinex = pd.read_parquet("../data/quinex/df_quinex.parquet")

In [ ]:
CHUNK_SIZE = 200_000
CHECKPOINT_DIR = "../data/quinex/checkpoints_new"
PARSE_TIMEOUT = 5  # seconds per row
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

total = len(df_quinex)
chunks_done = sorted([
    int(f.replace("chunk_", "").replace(".parquet", ""))
    for f in os.listdir(CHECKPOINT_DIR)
    if f.startswith("chunk_") and f.endswith(".parquet")
])

start_row = max(chunks_done) + CHUNK_SIZE if chunks_done else 0
print(f"Bereits abgeschlossene Chunks: {chunks_done}")
print(f"Starte bei Zeile {start_row} von {total}")

In [ ]:
for chunk_start in tqdm(range(start_row, total, CHUNK_SIZE), desc="Chunks"):
    chunk_end = min(chunk_start + CHUNK_SIZE, total)
    chunk = df_quinex.iloc[chunk_start:chunk_end].copy()

    # Process-basierter Timeout: hängende Parses werden nach PARSE_TIMEOUT Sekunden terminiert
    results = qpost.parse_batch_with_timeout(
        chunk["quantity_text"].tolist(),
        timeout=PARSE_TIMEOUT,
        desc=f"Rows {chunk_start}–{chunk_end}"
    )

    chunk["value_normalized_adapted"] = [r[0] for r in results]
    chunk["unit_normalized_adapted"] = [r[1] for r in results]
    chunk["quantity_normalized_adapted"] = [r[2] for r in results]

    # Serialisiere unit_normalized_adapted und quantity_normalized_adapted zu JSON-String (Dicts mit gemischten Typen sind nicht parquet-kompatibel)
    chunk["unit_normalized_adapted"] = chunk["unit_normalized_adapted"].apply(
        lambda x: json.dumps(x) if x is not None else None
    )
    chunk["quantity_normalized_adapted"] = chunk["quantity_normalized_adapted"].apply(
        lambda x: json.dumps(x) if x is not None else None
    )
    chunk.to_parquet(f"{CHECKPOINT_DIR}/chunk_{chunk_start}.parquet", index=True)

In [ ]:
# Alle Chunks speicherschonend zusammenführen
print("Füge alle Chunks zusammen...")
all_chunk_files = sorted(
    [f for f in os.listdir(CHECKPOINT_DIR) if f.startswith("chunk_") and f.endswith(".parquet")],
    key=lambda f: int(f.replace("chunk_", "").replace(".parquet", ""))
)

target_cols = [
    "value_normalized_adapted",
    "unit_normalized_adapted",
    "quantity_normalized_adapted",
]

for f in tqdm(all_chunk_files, desc="Chunks zusammenführen"):
    chunk_path = f"{CHECKPOINT_DIR}/{f}"
    chunk_df = pd.read_parquet(chunk_path)

    # JSON-Strings zurück in Python-Objekte konvertieren
    chunk_df["unit_normalized_adapted"] = chunk_df["unit_normalized_adapted"].apply(
        lambda x: json.loads(x) if isinstance(x, str) else None
    )
    chunk_df["quantity_normalized_adapted"] = chunk_df["quantity_normalized_adapted"].apply(
        lambda x: json.loads(x) if isinstance(x, str) else None
    )

    # Ergebnisse anhand des gespeicherten Index zurück in das Gesamtdatenframe schreiben
    df_quinex.loc[chunk_df.index, target_cols] = chunk_df[target_cols].values

print(f"Fertig! {len(df_quinex)} Zeilen verarbeitet.")

In [ ]:
# Nur die beiden Spalten serialisieren
df_quinex["unit_normalized_adapted"] = (
    df_quinex["unit_normalized_adapted"]
    .apply(lambda x: json.dumps(x) if x is not None else None)
)

df_quinex["quantity_normalized_adapted"] = (
    df_quinex["quantity_normalized_adapted"]
    .apply(lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x)
)

# Direkt speichern
df_quinex.to_parquet(
    "../data/quinex/df_quinex.parquet",
    index=False,
)

### **Rerun Unit Dimension**

In [ ]:
df_quinex = pd.read_parquet("../data/quinex/df_quinex.parquet")

In [ ]:
df_quinex = qpost.apply_unit_dimension_parsing(df_quinex)

In [ ]:
df_quinex.to_parquet("../data/quinex/df_quinex.parquet", index=False)

### **Add Author Country Info to Dataframe**

In [ ]:
df = pd.read_parquet("../data/paper/paper_dataset.parquet")
doi_liste = df["DOI"].tolist()

all_responses = []
for start in tqdm(range(0, len(doi_liste), 200), desc="DOI-Batches"):
    batch = doi_liste[start:start + 100]
    a = oalex.fetch_by_doi(batch)

    batch_response = oalex.extract_funding_and_institutions(
        a,
        output_path=f"../data/visualisation_files/oalex_data.json"
    )
    all_responses.extend(batch_response)

In [ ]:
df_quinex = pd.read_parquet("../data/quinex/df_quinex.parquet")
df_metadata = pd.read_parquet("../data/quinex/df_quinex_metadata.parquet")
df_paper = pd.read_parquet("../data/paper/paper_dataset.parquet")
with open("../data/visualisation_files/oalex_data.json", encoding="utf-8") as f:
    oalex_raw = json.load(f)

df_oalex = (
    pd.DataFrame.from_dict(oalex_raw, orient="index")
    .reset_index()
    .rename(columns={"index": "DOI"})
)

if "author_origins" not in df_oalex.columns:
    df_oalex["author_origins"] = [[] for _ in range(len(df_oalex))]

In [ ]:
# author_countries aus OpenAlex ersetzen und publisher aus paper ergänzen
def normalize_doi(value):
    if pd.isna(value):
        return None
    doi = str(value).strip().lower()
    doi = doi.replace("https://doi.org/", "").replace("http://doi.org/", "").replace("doi.org/", "")
    if doi.startswith("doi:"):
        doi = doi[4:]
    return doi

df_quinex = df_quinex.copy()
df_quinex["_doi_key"] = df_quinex["DOI"].apply(normalize_doi)

if "author_countries" in df_quinex.columns:
    df_quinex = df_quinex.drop(columns=["author_countries"])

if "author_origins" not in df_oalex.columns:
    raise KeyError("Spalte 'author_origins' fehlt in df_oalex.")
if "Publisher" not in df_paper.columns:
    raise KeyError("Spalte 'Publisher' fehlt in df_paper.")

oalex_lookup = (
    df_oalex[["DOI", "author_origins"]]
    .copy()
    .assign(_doi_key=lambda d: d["DOI"].apply(normalize_doi))
    .drop_duplicates(subset=["_doi_key"], keep="first")
    [["_doi_key", "author_origins"]]
    .rename(columns={"author_origins": "author_countries"})
)

paper_lookup = (
    df_paper[["DOI", "Publisher"]]
    .copy()
    .assign(_doi_key=lambda d: d["DOI"].apply(normalize_doi))
    .drop_duplicates(subset=["_doi_key"], keep="first")
    [["_doi_key", "Publisher"]]
    .rename(columns={"Publisher": "publisher"})
)

df_quinex = df_quinex.merge(oalex_lookup, on="_doi_key", how="left")
df_quinex = df_quinex.merge(paper_lookup, on="_doi_key", how="left")
df_quinex = df_quinex.drop(columns=["_doi_key"])

print(df_quinex[["author_countries", "publisher"]].notna().mean().rename("fill_rate"))
df_quinex.head(2)

In [ ]:
df_quinex.to_parquet("../data/quinex/df_quinex.parquet", index=False)

### **Adapt Column Types**

In [ ]:
df_quinex = pd.read_parquet("../data/quinex/df_quinex.parquet")

In [ ]:
print('------------------------------')
print(df_quinex['unit_normalized'].apply(type).value_counts())
print('------------------------------')
print(df_quinex['quantity_normalized'].apply(type).value_counts())

In [ ]:
# quantity_normalized: str → dict
df_quinex['quantity_normalized'] = df_quinex['quantity_normalized'].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)


df_quinex['unit_normalized'] = df_quinex['unit_normalized'].apply(
    lambda x: np.array(json.loads(x)) if isinstance(x, str) else np.array(x) if isinstance(x, list) else x
)

default_unit = np.array([{'exponent': None, 'text': None, 'unit_uri': 'http://qudt.org/vocab/unit/UNITLESS', 'year': None}])
df_quinex['unit_normalized'] = df_quinex['unit_normalized'].apply(
    lambda x: default_unit if x is None else x
)

In [ ]:
df_quinex['quantity_normalized'] = df_quinex['quantity_normalized'].apply(
    lambda x: json.dumps(x) if isinstance(x, dict) else x
)

In [ ]:
df_quinex.to_parquet("../data/quinex/df_quinex.parquet", index=False)

### **Rename Columns**

In [ ]:
df_quinex = pd.read_parquet("../data/quinex/df_quinex.parquet")

In [ ]:
df_quinex.columns

In [ ]:
# Spalten entfernen
df_quinex = df_quinex.drop(columns=[
    'spatial_scope',
    'value_normalized',
    'unit_normalized',
    'quantity_normalized'
])

# Spalten umbenennen
df_quinex = df_quinex.rename(columns={
    'spatial_scope_refined': 'spatial_scope',
    'value_normalized_adapted': 'value_normalized',
    'unit_normalized_adapted': 'unit_normalized',
    'quantity_normalized_adapted': 'quantity_normalized',
    'unit_dimension_adapted': 'unit_dimension'
})

In [ ]:
# 3) gewünschte Reihenfolge definieren
new_order = [
    "DOI",
    "source_url",
    #"Abstract",
    "open_access",
    "license_information",
    "publisher",
    "publication_year",
    "author",
    "author_countries",
    "entity",
    "entity_position",
    "property",
    "property_position",
    "quantity_text",
    "quantity_position",
    "quote",
    "quote_position",
    "value_normalized",
    "unit_normalized",
    "unit_dimension",
    "quantity_normalized",
    "temporal_scope",
    "temporal_scope_position",
    "spatial_scope",
    "spatial_scope_position",
    "reference",
    "reference_position",
    "method",  
    "method_position",
    "qualifier",
    "qualifier_position",
    "classification"
]

# 4) nur vorhandene Spalten berücksichtigen (robust gegen Missing Columns)
df_quinex = df_quinex[[col for col in new_order if col in df_quinex.columns]]

In [ ]:
df_quinex.columns

In [ ]:
df_quinex.to_parquet("../data/quinex/df_quinex.parquet", index=False)

## **Metadata Dataframe**

### **Add SJR to Metadata**

In [ ]:
metadata_df = pd.read_parquet("../data/quinex/df_quinex_metadata.parquet")
df_paper = pd.read_parquet("../data/paper/paper_dataset.parquet")
df_paper = df_paper[df_paper["Quinex"] == True]

In [ ]:
 metadata_df = metadata.add_sjr_to_metadata(
    metadata_df,
    sjr_folder="../data/paper/impactfactor",
    source_col="source",
    year_col="publication_year",
    #source_rename_dict=source_rename_dict
)

metadata_df = metadata_df.merge(
    df_paper[['DOI', 'Source title']],
    on='DOI',
    how='left'
)

mask = metadata_df['SJR'].isna()
metadata_fallback = metadata_df.loc[mask].copy()
metadata_fallback = metadata.add_sjr_to_metadata(
    metadata_fallback,
    sjr_folder="../data/paper/impactfactor",
    source_col="Source title",
    year_col="publication_year",
)
metadata_df.loc[mask, 'SJR'] = metadata_fallback['SJR'].values

In [ ]:
metadata_df.to_parquet("../data/quinex/df_quinex_metadata.parquet")